In [1]:
import gensim.downloader as api
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load models
model_en = api.load("glove-wiki-gigaword-100") 

# model_es = api.load("fasttext-wiki-news-subwords-300-es")
# model_ru = api.load("fasttext-wiki-news-subwords-300-ru")

In [2]:
def get_gender_direction(model, male_words, female_words):
    differences = []
    for m, f in zip(male_words, female_words):
        if m in model and f in model:
            differences.append(model[m] - model[f])
    return np.mean(differences, axis=0)

# English Anchors
en_m = ['man', 'male', 'boy', 'father']
en_f = ['woman', 'female', 'girl', 'mother']
gender_vector_en = get_gender_direction(model_en, en_m, en_f)

# Spanish Anchors
es_m = ['hombre', 'varón', 'niño', 'padre']
es_f = ['mujer', 'hembra', 'niña', 'madre']

# Russian Anchors
ru_m = ['мужчина', 'мужской', 'мальчик', 'отец']
ru_f = ['женщина', 'женский', 'девочка', 'мать']

In [3]:
def calculate_bias(model, target_word, gender_vector):
    if target_word in model:
        target_vec = model[target_word]
        # Cosine similarity
        bias_score = cosine_similarity(
            target_vec.reshape(1, -1), 
            gender_vector.reshape(1, -1)
        )[0][0]
        return bias_score
    return None

# Example test:
score = calculate_bias(model_en, "doctor", gender_vector_en)

test_words = ['doctor', 'nurse', 'authority', 'president']

{word: calculate_bias(model_en, word, gender_vector_en) for word in test_words}

{'doctor': np.float32(-0.06677766),
 'nurse': np.float32(-0.32890168),
 'authority': np.float32(0.09276364),
 'president': np.float32(0.12741259)}